# Planarian historical three-stage workflow

This notebook documents the original **planarian** SMED/SMESG workflow. It is separate from the human pando+GRN tutorial and its GLI3 comparison. The original stage scripts and their dependencies are preserved under `src/legacy/original_project/grnm-0.3/`. This notebook inspects the included small inputs; it does not train the historical models by default.


In [1]:
from pathlib import Path
import pandas as pd
ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'dataset').is_dir() and (p/'src').is_dir()),None)
if ROOT is None: raise RuntimeError('Start Jupyter inside the repository')
SOURCE=ROOT/'src/legacy/original_project/grnm-0.3'
DATA=ROOT/'dataset/legacy/planarian'
for label, filename in [('Phase 1','phase1_grn_z.py'),('Phase 2','phase2_rna2z.py'),('Phase 3','phase3_edge_predict.py')]:
 path=SOURCE/'scripts'/filename
 print(label,':',path.relative_to(ROOT),'available:',path.is_file())


Phase 1 : src/legacy/original_project/grnm-0.3/scripts/phase1_grn_z.py available: True
Phase 2 : src/legacy/original_project/grnm-0.3/scripts/phase2_rna2z.py available: True
Phase 3 : src/legacy/original_project/grnm-0.3/scripts/phase3_edge_predict.py available: True


## Included planarian inputs

The saved files include planarian ATAC PCA, network edges, TF annotation, SMED–SMESG mapping and gene annotation. The 66 MiB `data/rna_pca_new.csv` file is not included in the upload. Its local backup is documented in `dataset/legacy/planarian/README.md`; its public download location has not been established.


In [2]:
mapping=pd.read_csv(DATA/'data/smed_smesg_mapping.csv')
edges=pd.read_csv(DATA/'example_data/edges.csv',index_col=0)
print('SMED–SMESG mapping rows:',len(mapping))
print('Network edge rows:',len(edges))
print(mapping.head().to_string(index=False))
print('Large RNA PCA available:',(DATA/'data/rna_pca_new.csv').is_file())


SMED–SMESG mapping rows: 27542
Network edge rows: 11205
     smed_id       smesg_id
SMED30000001 SMESG000015275
SMED30000004 SMESG000039109
SMED30000005 SMESG000072036
SMED30000006 SMESG000050992
SMED30000008 SMESG000071901
Large RNA PCA available: False


## Three stages

1. `phase1_grn_z.py` combines RNA/ATAC features and network edges to construct gene embeddings.
2. `phase2_rna2z.py` trains a model to map RNA features to embeddings.
3. `phase3_edge_predict.py` trains a seven-class edge model and predicts candidate edges.

The historic code depends on DGL and other packages recorded in `requirements-legacy-planarian.txt`; this file is a historical version record, not a tested installation lockfile. The workflow has not been rerun end to end from this package. The source scripts were retained in their original layout and may contain historical assumptions.


## Optional: prepare a separate working directory

After obtaining `data/rna_pca_new.csv` and installing a compatible environment, run these commands from the repository root. They copy the code and inputs into a writable working directory; they do not alter the archival source.

```bash
mkdir -p results/generated/planarian_legacy
cp -a src/legacy/original_project/grnm-0.3/. results/generated/planarian_legacy/
cp -a dataset/legacy/planarian/. results/generated/planarian_legacy/
cd results/generated/planarian_legacy
python scripts/phase1_grn_z.py
python scripts/phase2_rna2z.py
python scripts/phase3_edge_predict.py
```

Phase 3 considers all gene pairs by default and may produce a very large output. Its `--max-pairs` option can be used for a smaller exploratory run; such an output is incomplete and must not be treated as the complete result. These commands require the missing RNA PCA input and have not been validated end to end in this package.
